In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes rasterio datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import os, json, math, time, zlib, re
import numpy as np, pandas as pd, torch, requests
from PIL import Image
from contextlib import nullcontext
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

SEED = 0
OUT = "/content/out"; os.makedirs(f"{OUT}/images", exist_ok=True)
ADAPTER_DIR = "/content/adapter"   # upload adapter_config.json + adapter_model.safetensors here

model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)
base = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
    device_map="auto")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

... (6 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [ ]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq

path = hf_hub_download(repo_id="BIFOLD-BigEarthNetv2-0/BigEarthNet.txt",
                       filename="BigEarthNet.txt.parquet", repo_type="dataset")
cols = ["ID", "patch_id", "input", "output", "type", "category", "split"]
kept = []
for batch in pq.ParquetFile(path).iter_batches(batch_size=50000, columns=cols):
    df = batch.to_pandas()
    df = df[(df["split"] == "test") & df["type"].isin(["binary", "mcq"])]
    if df.empty: continue
    df = df[df["patch_id"].map(lambda s: zlib.crc32(s.encode()) % 50 == 0)]
    if len(df): kept.append(df)
pool = pd.concat(kept, ignore_index=True)

rng = np.random.default_rng(SEED)
patches = rng.choice(pool["patch_id"].unique(), size=120, replace=False)
sub = pool[pool["patch_id"].isin(patches)].sample(frac=1, random_state=SEED)
test_df = sub.groupby("patch_id").head(4).reset_index(drop=True)
test_df["qid"] = test_df.index          # permanent question ID, used later for pairing
print(len(test_df), test_df["type"].value_counts().to_dict())
print(test_df[test_df["type"] == "binary"]["output"].str.lower().value_counts().to_dict())

BigEarthNet.txt.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

BigEarthNet.txt.parquet: downloading bytes:           |  0.00B            

480 {'mcq': 241, 'binary': 239}
{'no': 132, 'yes': 107}


In [ ]:
from datasets import load_dataset
from rasterio.io import MemoryFile

targets = set(test_df["patch_id"])
ds = load_dataset("hackelle/BigEarthNetV2-LMDB", split="all_data", streaming=True)
s2v1 = {}
for item in ds:
    if item["patch_id"] in targets:
        s2v1[item["patch_id"]] = item["s2v1_name"]
        if len(s2v1) == len(targets): break
print("mapped", len(s2v1), "/", len(targets))

def band(name, b, tries=3):
    url = f"https://data.source.coop/tu-berlin/bigearthnet/{name}/{name}_{b}.tif"
    for _ in range(tries):
        try:
            r = requests.get(url, timeout=20); r.raise_for_status()
            with MemoryFile(r.content) as mf:
                with mf.open() as d: return d.read(1)
        except Exception:
            time.sleep(2)
    raise RuntimeError(f"failed {name} {b}")

def stretch(x):
    lo, hi = np.percentile(x, (2, 98))
    return (np.clip((x - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

paths = {}
for i, (pid, name) in enumerate(s2v1.items(), 1):
    f = f"{OUT}/images/{pid}.jpg"
    if not os.path.exists(f):
        try:
            rgb = np.dstack([stretch(band(name, b)) for b in ("B04", "B03", "B02")])
            Image.fromarray(rgb).save(f)      # default JPEG settings, same as the original run
        except Exception as e:
            print("FAILED", pid, e); continue
    paths[pid] = f
    if i % 20 == 0: print(f"{i}/{len(s2v1)}")

test_df = test_df[test_df["patch_id"].isin(paths)].reset_index(drop=True)
test_df.to_csv(f"{OUT}/test_df.csv", index=False)
print(len(test_df), "questions on", test_df["patch_id"].nunique(), "patches")

README.md:   0%|          | 0.00/7.38k [00:00<?, ?B/s]

mapped 120 / 120
20/120
40/120
60/120
80/120
100/120
120/120
480 questions on 120 patches


In [ ]:
from peft import PeftModel
print(open(f"{ADAPTER_DIR}/adapter_config.json").read()[:700])
print(round(os.path.getsize(f"{ADAPTER_DIR}/adapter_model.safetensors") / 1e6, 1), "MB")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": {
    "base_model_class": "LlavaForConditionalGeneration",
    "parent_library": "transformers.models.llava.modeling_llava"
  },
  "base_model_name_or_path": "llava-hf/llava-1.5-7b-hf",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga
20.0 MB


PeftModel(
  (base_model): LoraModel(
    (model): LlavaForConditionalGeneration(
      (model): LlavaModel(
        (vision_tower): CLIPVisionModel(
          (embeddings): CLIPVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
            (position_embedding): Embedding(577, 1024)
          )
          (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (encoder): CLIPEncoder(
            (layers): ModuleList(
              (0-23): 24 x CLIPEncoderLayer(
                (self_attn): CLIPAttention(
                  (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (v_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(


In [ ]:
import os
from google.colab import files

os.makedirs("/content/adapter", exist_ok=True)
up = files.upload()
for name, data in up.items():
    dest = "adapter_config.json" if "config" in name else \
           "adapter_model.safetensors" if name.endswith(".safetensors") else name
    open(f"/content/adapter/{dest}", "wb").write(data)

print(os.listdir("/content/adapter"))
print(round(os.path.getsize("/content/adapter/adapter_model.safetensors") / 1e6, 1), "MB")

Saving adapter_model.safetensors to adapter_model.safetensors
Saving adapter_config.json to adapter_config.json
['adapter_config.json', 'adapter_model.safetensors']
20.0 MB


In [ ]:
print(len(paths), "patches |", len(test_df), "questions")

120 patches | 480 questions


In [ ]:
lora = run_condition("lora")
for t, g in lora.groupby("type"):
    print(t, int(g["correct"].sum()), "/", len(g))
print("the paper's adapter scored: binary 123/239, mcq 76/241")

NameError: name 'run_condition' is not defined

In [ ]:
print(len(paths), "patches |", len(test_df), "questions")

120 patches | 480 questions


In [ ]:
def parse(ans, kind):
    a = ans.strip().lower()
    m = re.match(r"(yes|no)\b", a) if kind == "binary" else re.match(r"\(?([abcd])\b", a)
    return m.group(1) if m else None

INSTR = {"binary": " Answer with only 'yes' or 'no'.",
         "mcq": " Answer with only the letter of the correct option."}

def run_condition(name, use_adapter=True, blank=False, instruct=False):
    out_csv = f"{OUT}/pred_{name}.csv"
    if os.path.exists(out_csv):
        print("already done:", name); return pd.read_csv(out_csv)
    rows, t0 = [], time.time()
    for k, r in test_df.iterrows():
        img = (Image.new("RGB", (336, 336), (128, 128, 128)) if blank
               else Image.open(paths[r["patch_id"]]).convert("RGB"))
        q = r["input"] + (INSTR[r["type"]] if instruct else "")
        inp = processor(text=f"USER: <image>\n{q}\nASSISTANT:", images=img, return_tensors="pt").to(model.device)
        ctx = nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.no_grad():
            out = model.generate(**inp, max_new_tokens=8, do_sample=False)
        raw = processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        gt, pred = str(r["output"]).strip().lower(), parse(raw, r["type"])
        rows.append(dict(qid=r["qid"], patch_id=r["patch_id"], type=r["type"], gt=gt, raw=raw, pred=pred, correct=int(pred == gt)))
        if (k + 1) % 50 == 0: print(f"{name}: {k+1}/{len(test_df)}  {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False); return df

In [ ]:
lora = run_condition("lora")
for t, g in lora.groupby("type"):
    print(t, int(g["correct"].sum()), "/", len(g))
print("the paper's adapter scored: binary 123/239, mcq 76/241")

lora: 50/480  42s
lora: 100/480  83s
lora: 150/480  123s
lora: 200/480  166s
lora: 250/480  207s
lora: 300/480  248s
lora: 350/480  290s
lora: 400/480  332s
lora: 450/480  373s
binary 110 / 239
mcq 70 / 241
the paper's adapter scored: binary 123/239, mcq 76/241


In [ ]:
lora = pd.read_csv(f"{OUT}/pred_lora.csv")
print(lora.groupby("type")["pred"].value_counts(dropna=False).to_string())
print(lora[["type", "gt", "raw"]].head(8).to_string())

type    pred
binary  yes     214
        no       25
mcq     d        79
        b        77
        c        55
        a        30
     type   gt                         raw
0     mcq    d                           b
1     mcq    b                           b
2  binary   no                         yes
3     mcq    a                           b
4     mcq    c  a) Broad-leaved forest and
5  binary   no                         yes
6     mcq    b                           d
7  binary  yes                         yes


In [ ]:
chunks, n = [], 0
for batch in pq.ParquetFile(path).iter_batches(batch_size=50000, columns=cols):
    df = batch.to_pandas()
    df = df[(df["split"] == "test") & df["type"].isin(["binary", "mcq"])]
    chunks.append(df); n += len(df)
    if n >= 100: break
intile_df = pd.concat(chunks, ignore_index=True).head(100).reset_index(drop=True)
intile_df["qid"] = intile_df.index
print(intile_df["type"].value_counts().to_dict(), "|", intile_df["patch_id"].nunique(), "patches")
print(intile_df[intile_df["type"] == "binary"]["output"].str.lower().value_counts().to_dict())

{'binary': 54, 'mcq': 46} | 7 patches
{'yes': 27, 'no': 27}


In [ ]:
os.rename(f"{OUT}/pred_lora.csv", f"{OUT}/pred_a400_lora.csv")

In [ ]:
base_instr = run_condition("base_instr", use_adapter=False, instruct=True)
base = run_condition("base", use_adapter=False)

base_instr: 50/480  43s
base_instr: 100/480  86s
base_instr: 150/480  125s
base_instr: 200/480  167s
base_instr: 250/480  209s
base_instr: 300/480  249s
base_instr: 350/480  292s
base_instr: 400/480  335s
base_instr: 450/480  377s
base: 50/480  53s
base: 100/480  103s
base: 150/480  156s
base: 200/480  207s
base: 250/480  259s
base: 300/480  311s
base: 350/480  363s
base: 400/480  416s
base: 450/480  470s


In [ ]:
print(sorted(os.listdir(OUT)))
ref = {"base": (105, 58), "base_instr": (108, 59)}   # earlier run, binary/241 and mcq/241
for n in ["base", "base_instr"]:
    d = pd.read_csv(f"{OUT}/pred_{n}.csv")
    b = int(d[d["type"] == "binary"]["correct"].sum()); m = int(d[d["type"] == "mcq"]["correct"].sum())
    print(f"{n}: binary {b}/239 (earlier {ref[n][0]}) | mcq {m}/241 (earlier {ref[n][1]})")

['images', 'pred_a400_lora.csv', 'pred_base.csv', 'pred_base_instr.csv', 'test_df.csv']
base: binary 105/239 (earlier 105) | mcq 58/241 (earlier 58)
base_instr: binary 108/239 (earlier 108) | mcq 59/241 (earlier 59)


In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", OUT)
from google.colab import files; files.download("/content/results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
shutil.make_archive("/content/results_a400", "zip", OUT)
from google.colab import files; files.download("/content/results_a400.zip")

base = model.unload()          # removes the adapter layers, giving the plain 4-bit model back
del model; torch.cuda.empty_cache()
print("leftover LoRA modules:", sum("lora" in n for n, _ in base.named_modules()))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

leftover LoRA modules: 0


In [ ]:
path = hf_hub_download(repo_id="BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", filename="BigEarthNet.txt.parquet", repo_type="dataset")
cols = ["ID", "patch_id", "input", "output", "type", "category", "split"]
kept = []
for batch in pq.ParquetFile(path).iter_batches(batch_size=50000, columns=cols):
    df = batch.to_pandas()
    df = df[(df["split"] == "train") & df["type"].isin(["binary", "mcq"])]
    if df.empty: continue
    df = df[df["patch_id"].map(lambda s: zlib.crc32(s.encode()) % 50 == 0)]
    if len(df): kept.append(df)
pool_tr = pd.concat(kept, ignore_index=True)
ids = np.random.default_rng(1).choice(pool_tr["patch_id"].unique(), size=300, replace=False)
sub = pool_tr[pool_tr["patch_id"].isin(ids)].sample(frac=1, random_state=1)
train_df = sub.groupby("patch_id").head(8).reset_index(drop=True)
assert not set(train_df["patch_id"]) & set(test_df["patch_id"]), "train/test patch overlap!"
print(len(train_df), train_df["type"].value_counts().to_dict(), train_df[train_df["type"]=="binary"]["output"].str.lower().value_counts().to_dict())

2400 {'binary': 1240, 'mcq': 1160} {'no': 636, 'yes': 604}


In [ ]:
def fetch_images(df, tag):
    targets = set(df["patch_id"])
    ds = load_dataset("hackelle/BigEarthNetV2-LMDB", split="all_data", streaming=True)
    m = {}
    for item in ds:
        if item["patch_id"] in targets:
            m[item["patch_id"]] = item["s2v1_name"]
            if len(m) == len(targets): break
    out = {}
    for i, (pid, name) in enumerate(m.items(), 1):
        f = f"{OUT}/images/{pid}.jpg"
        if not os.path.exists(f):
            try:
                Image.fromarray(np.dstack([stretch(band(name, b)) for b in ("B04", "B03", "B02")])).save(f)
            except Exception as e:
                print("FAILED", pid, e); continue
        out[pid] = f
        if i % 50 == 0: print(f"{tag}: {i}/{len(m)}")
    return out

train_paths = fetch_images(train_df, "train")
train_df = train_df[train_df["patch_id"].isin(train_paths)].reset_index(drop=True)
train_df.to_csv(f"{OUT}/train_df.csv", index=False)
print(len(train_df), "questions on", train_df["patch_id"].nunique(), "patches")

train: 50/300
train: 100/300
train: 150/300
train: 200/300
train: 250/300
train: 300/300
2400 questions on 300 patches


In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class VQADS(Dataset):
    def __init__(self, df, pth): self.df, self.pth = df.reset_index(drop=True), pth
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(self.pth[r["patch_id"]]).convert("RGB")
        prompt = f"USER: <image>\n{r['input']}\nASSISTANT:"
        p = processor(text=prompt, images=img, return_tensors="pt")
        f = processor(text=prompt + f" {r['output']}", images=img, return_tensors="pt")
        labels = f["input_ids"].clone()
        labels[:, :p["input_ids"].shape[1]] = -100      # prompt + image tokens are ignored
        f["labels"] = labels
        return {k: v.squeeze(0) for k, v in f.items()}

train_ds = VQADS(train_df, train_paths)
for i in range(3):
    s = train_ds[i]; sup = s["labels"][s["labels"] != -100]
    print(repr(processor.tokenizer.decode(sup)), "| gt:", train_df.iloc[i]["output"], "|", len(sup), "supervised of", len(s["labels"]), "tokens")

NameError: name 'train_df' is not defined

In [ ]:
import os
for n in ["processor", "base", "test_df", "paths", "band", "stretch", "run_condition", "train_df", "train_paths"]:
    print(n, n in globals())
print(os.path.exists("/content/out") and sorted(os.listdir("/content/out")))

processor False
base False
test_df False
paths False
band False
stretch False
run_condition False
train_df False
train_paths False
False


In [ ]:
  # RUMTIME GOT RESTARTED SO HERE WE GO

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes rasterio datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import os, json, math, time, zlib, re, shutil, random
import numpy as np, pandas as pd, torch, requests
from PIL import Image
from contextlib import nullcontext
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
from datasets import load_dataset
from rasterio.io import MemoryFile
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

SEED = 0
OUT = "/content/out"; os.makedirs(f"{OUT}/images", exist_ok=True)

model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)
base = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True),
    device_map="auto")

def band(name, b, tries=3):
    url = f"https://data.source.coop/tu-berlin/bigearthnet/{name}/{name}_{b}.tif"
    for _ in range(tries):
        try:
            r = requests.get(url, timeout=20); r.raise_for_status()
            with MemoryFile(r.content) as mf:
                with mf.open() as d: return d.read(1)
        except Exception:
            time.sleep(2)
    raise RuntimeError(f"failed {name} {b}")

def stretch(x):
    lo, hi = np.percentile(x, (2, 98))
    return (np.clip((x - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def parse(ans, kind):
    a = ans.strip().lower()
    m = re.match(r"(yes|no)\b", a) if kind == "binary" else re.match(r"\(?([abcd])\b", a)
    return m.group(1) if m else None

INSTR = {"binary": " Answer with only 'yes' or 'no'.",
         "mcq": " Answer with only the letter of the correct option."}

def run_condition(name, use_adapter=True, blank=False, instruct=False):
    out_csv = f"{OUT}/pred_{name}.csv"
    if os.path.exists(out_csv):
        print("already done:", name); return pd.read_csv(out_csv)
    rows, t0 = [], time.time()
    for k, r in test_df.iterrows():
        img = (Image.new("RGB", (336, 336), (128, 128, 128)) if blank
               else Image.open(paths[r["patch_id"]]).convert("RGB"))
        q = r["input"] + (INSTR[r["type"]] if instruct else "")
        inp = processor(text=f"USER: <image>\n{q}\nASSISTANT:", images=img, return_tensors="pt").to(model.device)
        ctx = nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.no_grad():
            out = model.generate(**inp, max_new_tokens=8, do_sample=False)
        raw = processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        gt, pred = str(r["output"]).strip().lower(), parse(raw, r["type"])
        rows.append(dict(qid=r["qid"], patch_id=r["patch_id"], type=r["type"], gt=gt, raw=raw, pred=pred, correct=int(pred == gt)))
        if (k + 1) % 50 == 0: print(f"{name}: {k+1}/{len(test_df)}  {time.time()-t0:.0f}s")
    df = pd.DataFrame(rows); df.to_csv(out_csv, index=False); return df

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

... (6 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [ ]:
path = hf_hub_download(repo_id="BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", filename="BigEarthNet.txt.parquet", repo_type="dataset")
cols = ["ID", "patch_id", "input", "output", "type", "category", "split"]
kept = []
for batch in pq.ParquetFile(path).iter_batches(batch_size=50000, columns=cols):
    df = batch.to_pandas()
    df = df[(df["split"] == "test") & df["type"].isin(["binary", "mcq"])]
    if df.empty: continue
    df = df[df["patch_id"].map(lambda s: zlib.crc32(s.encode()) % 50 == 0)]
    if len(df): kept.append(df)
pool = pd.concat(kept, ignore_index=True)
rng = np.random.default_rng(SEED)
patches = rng.choice(pool["patch_id"].unique(), size=120, replace=False)
sub = pool[pool["patch_id"].isin(patches)].sample(frac=1, random_state=SEED)
test_df = sub.groupby("patch_id").head(4).reset_index(drop=True)
test_df["qid"] = test_df.index
test_df.to_csv(f"{OUT}/test_df.csv", index=False)
print(len(test_df), test_df["type"].value_counts().to_dict(), test_df[test_df["type"]=="binary"]["output"].str.lower().value_counts().to_dict())

BigEarthNet.txt.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

BigEarthNet.txt.parquet: downloading bytes:           |  0.00B            

480 {'mcq': 241, 'binary': 239} {'no': 132, 'yes': 107}


In [ ]:
from google.colab import files
files.upload()
!unzip -o results*.zip -d /content/out
test_df = pd.read_csv(f"{OUT}/test_df.csv")
paths = {p: f"{OUT}/images/{p}.jpg" for p in test_df["patch_id"].unique()}

Saving results_a400.zip to results_a400.zip
Saving results.zip to results.zip
Saving adapter_r1_2400_final.zip to adapter_r1_2400_final.zip
Archive:  results_a400.zip
caution: filename not matched:  results.zip


In [ ]:
import glob, zipfile
zips = glob.glob("/content/*.zip")
print(zips)
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print(os.path.basename(z), "|", len(names), "files | first:", names[:3])

['/content/results_a400.zip', '/content/adapter_r1_2400_final.zip', '/content/results.zip']
results_a400.zip | 125 files | first: ['images/', 'pred_base.csv', 'test_df.csv']
adapter_r1_2400_final.zip | 3 files | first: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']
results.zip | 125 files | first: ['images/', 'pred_base.csv', 'test_df.csv']


In [ ]:
import glob, zipfile, os, pandas as pd
OUT = "/content/out"
with zipfile.ZipFile("/content/results_a400.zip") as zf:
    zf.extractall(OUT)
print(sorted(os.listdir(OUT)))
print(len(os.listdir(f"{OUT}/images")), "images")

test_df = pd.read_csv(f"{OUT}/test_df.csv")
paths = {p: f"{OUT}/images/{p}.jpg" for p in test_df["patch_id"].unique()}
print(len(test_df), "questions |", len(paths), "patches | all images present:", all(os.path.exists(f) for f in paths.values()))
for n in ["base", "base_instr"]:
    d = pd.read_csv(f"{OUT}/pred_{n}.csv")
    print(n, "binary", int(d[d["type"] == "binary"]["correct"].sum()), "| mcq", int(d[d["type"] == "mcq"]["correct"].sum()))

['images', 'pred_a400_lora.csv', 'pred_base.csv', 'pred_base_instr.csv', 'test_df.csv']
120 images
480 questions | 120 patches | all images present: True
base binary 105 | mcq 58
base_instr binary 108 | mcq 59


In [ ]:
for n in ["processor", "base", "band", "stretch", "run_condition", "hf_hub_download", "pq", "load_dataset", "zlib", "SEED"]:
    print(n, n in globals())

processor True
base True
band True
stretch True
run_condition True
hf_hub_download True
pq True
load_dataset True
zlib True
SEED True


In [ ]:
path = hf_hub_download(repo_id="BIFOLD-BigEarthNetv2-0/BigEarthNet.txt", filename="BigEarthNet.txt.parquet", repo_type="dataset")
cols = ["ID", "patch_id", "input", "output", "type", "category", "split"]
kept = []
for batch in pq.ParquetFile(path).iter_batches(batch_size=50000, columns=cols):
    df = batch.to_pandas()
    df = df[(df["split"] == "train") & df["type"].isin(["binary", "mcq"])]
    if df.empty: continue
    df = df[df["patch_id"].map(lambda s: zlib.crc32(s.encode()) % 50 == 0)]
    if len(df): kept.append(df)
pool_tr = pd.concat(kept, ignore_index=True)
ids = np.random.default_rng(1).choice(pool_tr["patch_id"].unique(), size=300, replace=False)
sub = pool_tr[pool_tr["patch_id"].isin(ids)].sample(frac=1, random_state=1)
train_df = sub.groupby("patch_id").head(8).reset_index(drop=True)
assert not set(train_df["patch_id"]) & set(test_df["patch_id"]), "train/test patch overlap!"
print(len(train_df), train_df["type"].value_counts().to_dict(), train_df[train_df["type"]=="binary"]["output"].str.lower().value_counts().to_dict())

2400 {'binary': 1240, 'mcq': 1160} {'no': 636, 'yes': 604}


In [ ]:
def fetch_images(df, tag):
    targets = set(df["patch_id"])
    ds = load_dataset("hackelle/BigEarthNetV2-LMDB", split="all_data", streaming=True)
    m = {}
    for item in ds:
        if item["patch_id"] in targets:
            m[item["patch_id"]] = item["s2v1_name"]
            if len(m) == len(targets): break
    out = {}
    for i, (pid, name) in enumerate(m.items(), 1):
        f = f"{OUT}/images/{pid}.jpg"
        if not os.path.exists(f):
            try:
                Image.fromarray(np.dstack([stretch(band(name, b)) for b in ("B04", "B03", "B02")])).save(f)
            except Exception as e:
                print("FAILED", pid, e); continue
        out[pid] = f
        if i % 50 == 0: print(f"{tag}: {i}/{len(m)}")
    return out

train_paths = fetch_images(train_df, "train")
train_df = train_df[train_df["patch_id"].isin(train_paths)].reset_index(drop=True)
train_df.to_csv(f"{OUT}/train_df.csv", index=False)
print(len(train_df), "questions on", train_df["patch_id"].nunique(), "patches")

README.md:   0%|          | 0.00/7.38k [00:00<?, ?B/s]

train: 50/300
train: 100/300
train: 150/300
train: 200/300
train: 250/300
train: 300/300
2400 questions on 300 patches


In [ ]:
import random
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class VQADS(Dataset):
    def __init__(self, df, pth): self.df, self.pth = df.reset_index(drop=True), pth
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(self.pth[r["patch_id"]]).convert("RGB")
        prompt = f"USER: <image>\n{r['input']}\nASSISTANT:"
        p = processor(text=prompt, images=img, return_tensors="pt")
        f = processor(text=prompt + f" {r['output']}", images=img, return_tensors="pt")
        labels = f["input_ids"].clone()
        labels[:, :p["input_ids"].shape[1]] = -100      # prompt + image tokens are ignored
        f["labels"] = labels
        return {k: v.squeeze(0) for k, v in f.items()}

train_ds = VQADS(train_df, train_paths)
for i in range(3):
    s = train_ds[i]; sup = s["labels"][s["labels"] != -100]
    print(repr(processor.tokenizer.decode(sup)), "| gt:", train_df.iloc[i]["output"], "|", len(sup), "supervised of", len(s["labels"]), "tokens")

'b' | gt: b | 1 supervised of 618 tokens
'no' | gt: no | 1 supervised of 605 tokens
'no' | gt: no | 1 supervised of 599 tokens


In [ ]:
seed_all(SEED)
model = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none"))
model.print_trainable_parameters()      # expect 4,980,736

ACCUM = 4
g = torch.Generator().manual_seed(SEED)
opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
state = {"seen": 0, "loss": []}

import transformers, peft
json.dump(dict(seed=SEED, lr=1e-4, accum=ACCUM, rank=8, alpha=16, dropout=0.05, n_train=len(train_df),
               n_patches=int(train_df["patch_id"].nunique()), transformers=transformers.__version__, peft=peft.__version__,
               torch=torch.__version__, gpu=torch.cuda.get_device_name(0)), open(f"{OUT}/run_config.json", "w"), indent=1)

trainable params: 4,980,736 || all params: 7,068,407,808 || trainable%: 0.0705


In [ ]:
def save_ckpt(tag, download=False):
    d = f"{OUT}/adapter_{tag}"; model.save_pretrained(d); shutil.make_archive(d, "zip", d)
    if download:
        from google.colab import files; files.download(d + ".zip")

def train_epochs(k, max_min=120, tag="r1"):
    model.train(); t0 = time.time()
    def stream():
        for _ in range(k):
            yield from DataLoader(train_ds, batch_size=1, shuffle=True, generator=g)
    for batch in stream():
        batch = {a: b.to(model.device) for a, b in batch.items()}
        loss = model(**batch).loss
        if not torch.isfinite(loss):
            print("non-finite loss, sample skipped"); continue
        (loss / ACCUM).backward()
        state["seen"] += 1; state["loss"].append(loss.item())
        if state["seen"] % ACCUM == 0: opt.step(); opt.zero_grad()
        if state["seen"] % 100 == 0:
            print(f"{state['seen']} samples | mean loss (last 100) {np.mean(state['loss'][-100:]):.3f} | {time.time()-t0:.0f}s")
        if state["seen"] % 400 == 0: save_ckpt(f"{tag}_{state['seen']}")
        if time.time() - t0 > max_min * 60: print("time cap reached"); break
    if state["seen"] % ACCUM: opt.step(); opt.zero_grad()
    pd.DataFrame({"loss": state["loss"]}).to_csv(f"{OUT}/train_loss_{tag}.csv", index=False)
    save_ckpt(f"{tag}_{state['seen']}_final", download=True)

train_epochs(1)

100 samples | mean loss (last 100) 5.029 | 156s
200 samples | mean loss (last 100) 1.232 | 308s
300 samples | mean loss (last 100) 1.000 | 459s
400 samples | mean loss (last 100) 0.953 | 612s
500 samples | mean loss (last 100) 1.058 | 764s
600 samples | mean loss (last 100) 1.121 | 916s
700 samples | mean loss (last 100) 0.998 | 1068s
800 samples | mean loss (last 100) 1.013 | 1219s
900 samples | mean loss (last 100) 1.011 | 1372s
1000 samples | mean loss (last 100) 0.932 | 1525s
1100 samples | mean loss (last 100) 0.944 | 1677s
1200 samples | mean loss (last 100) 0.896 | 1828s
1300 samples | mean loss (last 100) 0.993 | 1980s
1400 samples | mean loss (last 100) 0.965 | 2132s
1500 samples | mean loss (last 100) 0.935 | 2284s
1600 samples | mean loss (last 100) 0.903 | 2436s
1700 samples | mean loss (last 100) 0.932 | 2590s
1800 samples | mean loss (last 100) 1.018 | 2741s
1900 samples | mean loss (last 100) 0.915 | 2892s
2000 samples | mean loss (last 100) 0.931 | 3044s
2100 samples | 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval();      # turns dropout off; the earlier evaluation cell did this once, and training switched it back on
r1_lora  = run_condition("r1_lora")
r1_blind = run_condition("r1_blind", blank=True)

r1_lora: 50/480  50s
r1_lora: 100/480  99s
r1_lora: 150/480  144s
r1_lora: 200/480  190s
r1_lora: 250/480  237s
r1_lora: 300/480  284s
r1_lora: 350/480  331s
r1_lora: 400/480  380s
r1_lora: 450/480  427s
r1_blind: 50/480  48s
r1_blind: 100/480  96s
r1_blind: 150/480  139s
r1_blind: 200/480  186s
r1_blind: 250/480  231s
r1_blind: 300/480  278s
r1_blind: 350/480  324s
r1_blind: 400/480  372s
r1_blind: 450/480  417s


In [ ]:
P = {"base": pd.read_csv(f"{OUT}/pred_base.csv"), "base_instr": pd.read_csv(f"{OUT}/pred_base_instr.csv"),
     "lora": pd.read_csv(f"{OUT}/pred_r1_lora.csv"), "blind": pd.read_csv(f"{OUT}/pred_r1_blind.csv")}

In [ ]:
try:
    print("train:", len(train_df), "questions |", train_df["patch_id"].nunique(), "patches | test:", len(test_df))
    print("samples trained:", state["seen"], "| mean loss, last 100:", round(float(np.mean(state["loss"][-100:])), 3))
except NameError as e:
    print("not defined yet:", e)
print(sorted(f for f in os.listdir(OUT) if f.startswith(("adapter_", "pred_", "train_loss", "run_config"))))

train: 2400 questions | 300 patches | test: 480
samples trained: 2400 | mean loss, last 100: 0.847
['adapter_r1_1200', 'adapter_r1_1200.zip', 'adapter_r1_1600', 'adapter_r1_1600.zip', 'adapter_r1_2000', 'adapter_r1_2000.zip', 'adapter_r1_2400', 'adapter_r1_2400.zip', 'adapter_r1_2400_final', 'adapter_r1_2400_final.zip', 'adapter_r1_400', 'adapter_r1_400.zip', 'adapter_r1_800', 'adapter_r1_800.zip', 'pred_a400_lora.csv', 'pred_base.csv', 'pred_base_instr.csv', 'pred_r1_blind.csv', 'pred_r1_lora.csv', 'run_config.json', 'train_loss_r1.csv']


In [ ]:
import zipfile, glob
keep = glob.glob(f"{OUT}/*.csv") + glob.glob(f"{OUT}/*.json") + [f"{OUT}/images/{p}.jpg" for p in test_df["patch_id"].unique()]
with zipfile.ZipFile("/content/results_r1.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep: z.write(f, os.path.relpath(f, OUT))
from google.colab import files
files.download("/content/results_r1.zip")
files.download(f"{OUT}/adapter_r1_2400_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import math
from scipy.stats import binomtest

def wilson(k, n, z=1.96):
    p = k/n; d = 1+z*z/n; c = (p+z*z/(2*n))/d
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return 100*(c-h), 100*(c+h)

P = {"base": pd.read_csv(f"{OUT}/pred_base.csv"), "base_instr": pd.read_csv(f"{OUT}/pred_base_instr.csv"),
     "lora": pd.read_csv(f"{OUT}/pred_r1_lora.csv"), "blind": pd.read_csv(f"{OUT}/pred_r1_blind.csv")}

for n, d in P.items():
    for t, g in d.groupby("type"):
        k, N = int(g["correct"].sum()), len(g); lo, hi = wilson(k, N)
        print(f"{n:11s} {t:6s} {k}/{N} = {100*k/N:.1f}%  [{lo:.1f}, {hi:.1f}]")
    print(f"{'':11s} binary answers:", P[n][P[n]['type'] == 'binary']['pred'].value_counts(dropna=False).to_dict())

yn = P["lora"][P["lora"]["type"] == "binary"]["gt"].value_counts()
print("always-no:", round(100*yn["no"]/yn.sum(), 1), "% | always-yes:", round(100*yn["yes"]/yn.sum(), 1), "%")

def paired(a, c, t):
    Da = P[a][P[a]["type"] == t].set_index("qid"); Dc = P[c][P[c]["type"] == t].set_index("qid").loc[Da.index]
    A, B = Da["correct"], Dc["correct"]
    oa, oc = int(((A == 1) & (B == 0)).sum()), int(((A == 0) & (B == 1)).sum())
    p = binomtest(oa, oa + oc, 0.5).pvalue if oa + oc else 1.0
    per = pd.DataFrame({"d": (A - B).values, "pid": Da["patch_id"].values}).groupby("pid")["d"].agg(["sum", "count"])
    rng = np.random.default_rng(0); diffs = []
    for _ in range(5000):
        s = per.sample(len(per), replace=True, random_state=int(rng.integers(10**9)))
        diffs.append(100 * s["sum"].sum() / s["count"].sum())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{a} vs {c} [{t}]: {100*A.mean():.1f}% vs {100*B.mean():.1f}% | {a}-only right: {oa}, {c}-only right: {oc} | McNemar p = {p:.3f} | patch-bootstrap 95% [{lo:.1f}, {hi:.1f}] pts")

for t in ("binary", "mcq"):
    paired("lora", "base_instr", t); paired("lora", "blind", t)

base        binary 105/239 = 43.9%  [37.8, 50.3]
base        mcq    58/241 = 24.1%  [19.1, 29.8]
            binary answers: {'yes': 201, 'no': 29, nan: 9}
base_instr  binary 108/239 = 45.2%  [39.0, 51.5]
base_instr  mcq    59/241 = 24.5%  [19.5, 30.3]
            binary answers: {'yes': 208, 'no': 31}
lora        binary 139/239 = 58.2%  [51.8, 64.2]
lora        mcq    111/241 = 46.1%  [39.9, 52.4]
            binary answers: {'no': 214, 'yes': 25}
blind       binary 137/239 = 57.3%  [51.0, 63.4]
blind       mcq    106/241 = 44.0%  [37.9, 50.3]
            binary answers: {'no': 226, 'yes': 13}
always-no: 55.2 % | always-yes: 44.8 %
lora vs base_instr [binary]: 58.2% vs 45.2% | lora-only right: 113, base_instr-only right: 82 | McNemar p = 0.031 | patch-bootstrap 95% [1.2, 23.9] pts
lora vs blind [binary]: 58.2% vs 57.3% | lora-only right: 9, blind-only right: 7 | McNemar p = 0.804 | patch-bootstrap 95% [-2.4, 4.1] pts
lora vs base_instr [mcq]: 46.1% vs 24.5% | lora-only right: 81, base

In [ ]:
d = P["lora"][P["lora"]["type"] == "binary"].set_index("qid")
a, c = d["correct"], (d["gt"] == "no").astype(int)
oa, oc = int(((a == 1) & (c == 0)).sum()), int(((a == 0) & (c == 1)).sum())
print("lora-only right:", oa, "| always-no-only right:", oc, "| McNemar p =", round(binomtest(oa, oa + oc, 0.5).pvalue, 3))

cat = test_df.set_index("qid")["category"]
for n in ["base_instr", "lora", "blind"]:
    P[n]["category"] = P[n]["qid"].map(cat)
tab = pd.concat({n: P[n].groupby("category")["correct"].mean() * 100 for n in ["base_instr", "lora", "blind"]}, axis=1)
tab["n"] = P["lora"].groupby("category").size()
print(tab.round(1).sort_values("n", ascending=False).to_string())

lora-only right: 16 | always-no-only right: 9 | McNemar p = 0.23
              base_instr  lora  blind    n
category                                  
count               33.7  66.3   67.3  101
area                34.3  45.5   44.4   99
presence            35.6  56.7   51.1   90
adjacency           45.5  49.4   46.8   77
season              25.0  52.3   50.0   44
climate zone        25.0  34.4   43.8   32
country             43.3  50.0   40.0   30
relative pos         0.0   0.0   14.3    7


In [ ]:
paths_orig = paths
perm = np.random.default_rng(SEED).permutation(test_df["patch_id"].unique())
swap = dict(zip(perm, np.roll(perm, 1)))          # every patch gets another patch's image
paths = {p: paths_orig[swap[p]] for p in perm}
r1_shuf = run_condition("r1_shuffled")
paths = paths_orig

P["shuf"] = pd.read_csv(f"{OUT}/pred_r1_shuffled.csv")
for t in ("binary", "mcq"): paired("lora", "shuf", t)

r1_shuffled: 50/480  50s
r1_shuffled: 100/480  100s
r1_shuffled: 150/480  144s
r1_shuffled: 200/480  190s
r1_shuffled: 250/480  237s
r1_shuffled: 300/480  284s
r1_shuffled: 350/480  331s
r1_shuffled: 400/480  380s
r1_shuffled: 450/480  426s
lora vs shuf [binary]: 58.2% vs 60.3% | lora-only right: 10, shuf-only right: 15 | McNemar p = 0.424 | patch-bootstrap 95% [-6.4, 2.1] pts
lora vs shuf [mcq]: 46.1% vs 43.6% | lora-only right: 15, shuf-only right: 9 | McNemar p = 0.307 | patch-bootstrap 95% [-1.3, 6.1] pts
